# qdmpy Developer Guide: Extending the Pipeline

**Target user**: "I want to develop my own algorithms."

Covers: adding a custom ESR model, adding a custom ODMR processor,
adding a custom field reconstructor, and using `FitManager` standalone.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import qdmpy
from qdmpy import (
    Model, ModelRegistry,
    Processor, ODMRData, ODMR,
    FieldReconstructor, MagneticMap, QDMResult,
    FitManager,
    make_synthetic_odmr_data, make_synthetic_fit_result,
)
from typing import ClassVar
from numpy.typing import NDArray

## 1. Custom ESR model

The built-in models (`ESR14N`, `ESR15N`, `ESRSINGLE`) use Lorentzian line shapes.
Here we add a **Gaussian** single-dip model to demonstrate the extension mechanism.

Requirements for a custom model:
- Subclass `Model`
- Set `name: ClassVar[str]` (used as the registry key)
- Call `super().__init__(name, n_peaks, parameter_names)` in `__init__`
- Set `model_id = -1` (CPU-only; gpufit is not used for custom models)
- Implement `parameter_types` property
- Implement `frequency_parameters` property
- Implement `func(x, parameters) -> NDArray` with shape `(N, n_freq)`

In [ ]:
@ModelRegistry.register
class GaussianSingle(Model):
    """Single-peak Gaussian dip model.

    Parameters: center (GHz), width (GHz), contrast (a.u.), offset (a.u.)
    """
    name: ClassVar[str] = 'GAUSSIAN'

    def __init__(self) -> None:
        super().__init__(
            'GAUSSIAN',
            n_peaks=1,
            parameter_names=['center', 'width', 'contrast', 'offset'],
        )
        self.model_id = -1  # CPU-only; gpufit not used for custom models

    @property
    def parameter_types(self) -> dict[str, str]:
        return {'center': 'center', 'width': 'width',
                'contrast': 'contrast', 'offset': 'offset'}

    @property
    def frequency_parameters(self) -> list[str]:
        return ['center']  # only center is stored in GHz units

    def func(self, x: NDArray, parameters: NDArray) -> NDArray:
        """Evaluate Gaussian dip spectrum.

        Args:
            x: Frequency values (n_freq,) in GHz.
            parameters: Shape (N, 4) — [center, width, contrast, offset].

        Returns:
            Fluorescence array of shape (N, n_freq).
        """
        parameters = np.atleast_2d(parameters)
        center   = parameters[:, 0:1]  # (N, 1)
        width    = parameters[:, 1:2]
        contrast = parameters[:, 2:3]
        offset   = parameters[:, 3:4]
        dip = contrast * np.exp(-0.5 * ((x - center) / width) ** 2)
        return 1.0 + offset - dip


print('Registered models:', ModelRegistry.available_models())

In [ ]:
# Verify: retrieve the model by name and inspect it
model = ModelRegistry.get('GAUSSIAN')
print('Name:             ', model.name)
print('n_peaks:          ', model.n_peaks)
print('parameter_names:  ', model.parameter_names)
print('units:            ', model.units)

In [ ]:
# Compare Gaussian vs built-in Lorentzian on the same parameters
from qdmpy.fitting.models import ESRSINGLE

freq = np.linspace(2.855, 2.895, 100)  # GHz
params = np.array([[2.875, 0.003, 0.2, 0.0]])  # (1, 4)

lorentz = ModelRegistry.get('ESRSINGLE')
gauss   = ModelRegistry.get('GAUSSIAN')

spec_lorentz = lorentz.func(freq, params).squeeze()
spec_gauss   = gauss.func(freq, params).squeeze()

print(f'Lorentzian output shape: {spec_lorentz.shape}')
print(f'Gaussian   output shape: {spec_gauss.shape}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(freq, spec_lorentz, label='Lorentzian (ESRSINGLE)')
ax.plot(freq, spec_gauss,   label='Gaussian (GAUSSIAN)', linestyle='--')
ax.set_xlabel('Frequency (GHz)')
ax.set_ylabel('Fluorescence')
ax.set_title('Line shape comparison')
ax.legend()
plt.tight_layout()
plt.savefig('lineshape_compare.png', dpi=80)
plt.show()

## 2. Custom ODMR processor

Processors implement the `Processor` protocol:
```python
class MyProcessor:
    def process(self, data: ODMRData) -> ODMRData: ...
    def describe(self) -> str: ...
```

Here we implement a **spectral clipping** processor that clamps fluorescence
values to [vmin, vmax] — useful for removing extreme outliers before fitting.

In [ ]:
class ClipProcessor:
    """Clamp fluorescence values to [vmin, vmax].

    Implements the Processor protocol — no base class required.
    """

    def __init__(self, vmin: float = 0.9, vmax: float = 1.1) -> None:
        self.vmin = vmin
        self.vmax = vmax

    def process(self, data: ODMRData) -> ODMRData:
        """Return a new ODMRData with clipped values."""
        clipped = data.data.clip(self.vmin, self.vmax)
        return ODMRData(data=clipped, metadata=data.metadata.copy())

    def describe(self) -> str:
        return f'ClipProcessor(vmin={self.vmin}, vmax={self.vmax})'


# Verify protocol conformance at runtime
print('Satisfies Processor protocol:', isinstance(ClipProcessor(), Processor))

In [ ]:
# Use it in the processing pipeline
from qdmpy import NormalizationProcessor

raw = make_synthetic_odmr_data(shape=(16, 16), noise=0.05)  # high noise
odmr = ODMR(raw)

odmr.processor_manager.add_processor(ClipProcessor(vmin=0.88, vmax=1.05))
odmr.processor_manager.add_processor(NormalizationProcessor())
odmr.process_data()

print('Pipeline:')
for p in odmr.processor_manager.processors:
    print(' ', p.describe())

In [ ]:
# Before / after comparison at one pixel
freq_raw, spec_raw   = odmr.spectrum(y=4, x=4, polarity='neg', freq_range='low', processed=False)
freq_proc, spec_proc = odmr.spectrum(y=4, x=4, polarity='neg', freq_range='low', processed=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 3), sharey=False)
axes[0].plot(freq_raw,  spec_raw,  'o-', ms=3)
axes[0].set_title('Raw (high noise)')
axes[1].plot(freq_proc, spec_proc, 'o-', ms=3, color='C1')
axes[1].set_title('After ClipProcessor + Normalisation')
for ax in axes:
    ax.set_xlabel('Frequency (GHz)')
    ax.set_ylabel('Fluorescence')
plt.tight_layout()
plt.savefig('clip_compare.png', dpi=80)
plt.show()

## 3. Custom field reconstructor

The default reconstruction (`MagneticMap`) applies Fourier-domain inversion
to convert B111 into (Bx, By, Bz).  Implement `FieldReconstructor` to replace
this step — for example, to use a different inversion method or a machine-learning
forward model.

```python
class MyReconstructor:
    def reconstruct(
        self,
        b111: xr.DataArray,              # dims (y, x), attrs['pixel_spacing']
        nv_axis: tuple[float, float, float],
    ) -> xr.Dataset:                     # must contain 'bx','by','bz','btotal'
        ...
```

In [ ]:
class DirectProjectionReconstructor:
    """Trivial reconstructor: project B111 onto NV axis, set Bx=By=0.

    This is physically inaccurate but demonstrates the interface.
    Replace the body with your own inversion algorithm.
    """

    def reconstruct(
        self,
        b111: xr.DataArray,
        nv_axis: tuple[float, float, float],
    ) -> xr.Dataset:
        nv = np.array(nv_axis)
        nv_norm = nv / np.linalg.norm(nv)

        # Project B111 onto NV axis components
        data = b111.values  # (H, W)
        bx = xr.DataArray(data * nv_norm[0], dims=b111.dims)
        by = xr.DataArray(data * nv_norm[1], dims=b111.dims)
        bz = xr.DataArray(data * nv_norm[2], dims=b111.dims)
        btotal = xr.DataArray(np.abs(data), dims=b111.dims)

        return xr.Dataset({'bx': bx, 'by': by, 'bz': bz, 'btotal': btotal})


# Runtime protocol check
print('Satisfies FieldReconstructor protocol:',
      isinstance(DirectProjectionReconstructor(), FieldReconstructor))

In [ ]:
# Pass custom reconstructor to QDMResult
from qdmpy.testing import make_synthetic_qdm_result

result_default = make_synthetic_qdm_result(shape=(32, 32))
result_custom  = QDMResult(
    fit_result=result_default.fit_result,
    reconstructor=DirectProjectionReconstructor(),
)

mm_default = result_default.magnetic_map
mm_custom  = result_custom.magnetic_map

print('Default Bz range: ', float(mm_default.bz.min()), '…', float(mm_default.bz.max()), 'µT')
print('Custom  Bz range: ', float(mm_custom.bz.min()),  '…', float(mm_custom.bz.max()),  'µT')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, mm, title in zip(axes, [mm_default, mm_custom], ['Default (Fourier)', 'Custom (projection)']):
    data = mm.bz.values
    vmax = np.percentile(np.abs(data), 98)
    im = ax.imshow(data, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(f'Bz — {title}')
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig('reconstructor_compare.png', dpi=80)
plt.show()

## 4. Using `FitManager` standalone

`FitManager` is stateless: it receives `ODMRData` and `frequencies`,
returns a `FitResult`, and can be reused with different data.

```python
fm = FitManager('ESR14N')         # or 'auto' to detect model from data
fit_result = fm.fit(
    odmr.processed_data.data,     # xr.DataArray
    odmr.processed_data.frequencies,  # NDArray (n_frange, n_freq)
    pixel_spacing=4e-6,
)
result = QDMResult(fit_result=fit_result)
```

Fitting requires `pyGpufit`.  Below we show the full workflow on synthetic
data so the notebook runs in CI; just replace `make_synthetic_fit_result`
with `fm.fit(...)` on a GPU machine.

In [ ]:
from qdmpy import make_synthetic_fit_result, is_pygpufit_available

print('pyGpufit available:', is_pygpufit_available())

# ── With GPU (uncomment on a machine with pyGpufit) ──────────────────────────
# raw = make_synthetic_odmr_data(shape=(32, 32))
# odmr = ODMR(raw)
# odmr.processor_manager.add_processor(NormalizationProcessor())
# odmr.process_data()
#
# fm = FitManager('ESR14N')        # 'auto' also works
# fit_result = fm.fit(
#     odmr.processed_data.data,
#     odmr.processed_data.frequencies,
#     pixel_spacing=odmr.processed_data.metadata.get('pixel_spacing', 4e-6),
# )
# ─────────────────────────────────────────────────────────────────────────────

# Without GPU: use synthetic fit result
fit_result = make_synthetic_fit_result(shape=(32, 32), model_name='ESR14N')

result = QDMResult(fit_result=fit_result)
print('b111_remanent shape:', result.b111_remanent.shape)
print('Model:              ', fit_result.model_name)
print('Pixel spacing (µm): ', fit_result.pixel_spacing * 1e6)

In [ ]:
# FitManager can be reused with different data
# (Shown conceptually — actual fitting needs GPU)
fm = FitManager('ESR14N')
print('FitManager ready:', fm)
print('GPU available:   ', is_pygpufit_available())

# Constraints can be set before fitting
# fm.set_constraints('width', vmin=0.001, vmax=0.01)

# Fit multiple datasets with the same FitManager:
# for dataset in datasets:
#     result = fm.fit(dataset.data, dataset.frequencies, pixel_spacing=4e-6)
#     results.append(result)

## Summary

| Extension point | Protocol / Base | Register via | Notes |
|----------------|-----------------|--------------|-------|
| Custom ESR model | `Model` (ABC) | `@ModelRegistry.register` | Set `model_id = -1` (CPU-only) |
| Custom processor | `Processor` (Protocol) | `odmr.processor_manager.add_processor()` | No inheritance needed |
| Custom reconstructor | `FieldReconstructor` (Protocol) | `QDMResult(reconstructor=...)` | Returns `xr.Dataset` with bx/by/bz/btotal |
| Standalone fitting | `FitManager` | Direct | Returns `FitResult`; requires pyGpufit |

See `docs/extending.md` for the full developer reference.